# Phase 01: Exploratory Data Analysis (EDA)

Based on the in initial strategy, this notebook will execute this steps:

* **Dataset Description:** Identify the target variable (`fraude`) and analyze its behavior and relationships with other features in the dataset.
* **Insight Extraction:** Identify statistical anomalies within variables, such as high-frequency categories or unusual temporal patterns.
* **Feature Selection:** Based on the EDA, determine which variables to keep, which to drop immediately, and identify opportunities for new features derived from the data distribution.


To accelerate the generation of the initial report, we used Pandas Profiling, which automatically produces descriptive statistics and relevant insights about the variables. After this first assessment, we were able to identify important patterns in the data and generate an initial list of insights on how each feature should be treated during preprocessing and feature engineering.

# Setup and Imports

In [5]:
from src.etl import load_raw_mercadolibre_dataset
from src.eda import generate_data_profiling


# Reading data

In [6]:
df_raw = load_raw_mercadolibre_dataset()
print("Data loaded:")
display(df_raw.head())

Loading raw MercadoLibre dataset from dataset\raw\MercadoLibre Data Scientist Technical Challenge - Dataset.csv...
Raw MercadoLibre dataset loaded successfully from dataset\raw\MercadoLibre Data Scientist Technical Challenge - Dataset.csv as a pandas DataFrame.
Data loaded:


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude
0,4,0.6812,50084.12,50.0,0.000000,20.0,AR,1,cat_d26ab52,0.365475,2479.0,952.0,1,NaN,Y,2020-03-20 09:28:19,57.63,100,0
1,4,0.6694,66005.49,0.0,0.000000,2.0,AR,1,cat_ea962fb,0.612728,2603.0,105.0,1,Y,Y,2020-03-09 13:58:28,40.19,25,0
2,4,0.4718,7059.05,4.0,0.463488,92.0,BR,25,cat_4c2544e,0.651835,2153.0,249.0,1,Y,Y,2020-04-08 12:25:55,5.77,23,0
3,4,0.7260,10043.10,24.0,0.046845,43.0,BR,43,cat_1b59ee3,0.692728,4845.0,141.0,1,N,Y,2020-03-14 11:46:13,40.89,23,0
4,4,0.7758,16584.42,2.0,0.154616,54.0,BR,0,cat_9bacaa5,0.201354,2856.0,18.0,1,Y,N,2020-03-23 14:17:13,18.98,71,0


# Generating Data Profiling 

In [7]:
# Directory to save the HTML report
generate_data_profiling(df_raw, "MercadoLibre Dataset Profile - Raw Data", "mercadolibre_profile_raw_data.html")



Generating data profiling report for MercadoLibre Dataset Profile - Raw Data...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:01<00:00,  9.99it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile report saved to: notebooks\results\mercadolibre_profile_raw_data.html


# Main Insights from Data Profiling

Based on the analysis of the dataset profile for the fraud detection project, here are the main highlights and statistical insights:


## **Dataset Overview**

* **Observations**: 150,000 rows.
* **Features**: 19 total, including numeric, categorical, boolean, and one DateTime variable.
* **Duplicates**: 0.0% (No duplicate rows identified).



## **Critical Data Imbalance & Missing Values**

* **Target Imbalance**: The target variable `fraude` is highly imbalanced, with **95.0%** non-fraudulent (0) and **5.0%** fraudulent (1) cases.
* **High Missingness**: Feature `o` has an extremely high missing rate of **72.6%** (108,857 values).
* **Moderate Missingness**: Features `b` and `c` both have **8.7%** missing values (12,984 values each).



## **Feature Statistics & Anomalies**

* **Unique Identifier**: Feature `k` contains 100% unique values, indicating it likely represents a transaction ID or primary key.

* **Zeros as Potential Indicators**:

  * `e`: 43.4% zeros
  * `f`: 16.9% zeros
  * `m`: 10.6% zeros
  * `h`: 8.6% zeros

  These concentrations suggest potential structural inactivity patterns rather than random zeros.

* **Extreme Skewness**:

  * `e`: extremely skewed (γ₁ ≈ 281.22)
  * `f`: highly skewed (γ₁ ≈ 126.18)

  This indicates heavy-tailed distributions where extreme values may carry important fraud signals.

* **Categorical Imbalance**:

  * Feature `a` is highly concentrated, with **85.7%** of observations in a single category (Value 4).

* **Monetary Distribution**:

  * `monto` presents a heavy-tailed distribution, meaning extreme transaction values may dominate financial impact.



## **Correlations**

* **Highly Correlated Feature Clusters**:

  * Cluster 1 (behavioral): `d` and `m` (with `p` related but categorical)
  * Cluster 2: `e` and `monto`
  * Cluster 3: `f` and `l`

These correlations may introduce multicollinearity issues in linear models.



# Summary of Identified Structural Issues

| Problem Identified                   | Variables Affected         | Technical Risk                                          | Business Justification                                                           |
| ------------------------------------ | -------------------------- | ------------------------------------------------------- | -------------------------------------------------------------------------------- |
| **Target imbalance (5% fraud)**      | fraude                     | Model biased toward majority class; accuracy misleading | Fraud cost is asymmetric (−100% vs +25%); decisions must reflect economic impact |
| **High missingness (72.6%)**         | o                          | Naive imputation may remove predictive signal           | Missingness may represent abnormal behavior                                      |
| **Moderate missingness (8.7%)**      | b, c                       | Loss of data or bias if dropped                         | Incomplete information may correlate with fraud risk                             |
| **Unique identifier (100% unique)**  | k                          | Leakage risk; no predictive value                       | Pure ID variable does not generalize                                             |
| **High zero concentration**          | e, f, m, h                 | Structural absence vs true zero unclear                 | Zero activity may indicate abnormal behavior                                     |
| **Extreme skewness**                 | e, f                       | Linear models unstable; extreme values dominate         | Tail behavior likely informative                                                 |
| **Skewed monetary variable**         | monto                      | Heavy tail; extreme losses dominate profit              | High-value fraud drives economic loss                                            |
| **Correlation clusters**             | (d, m), (e, monto), (f, l) | Multicollinearity in linear models                      | Avoid unstable coefficients                                                      |
| **Categorical imbalance**            | a                          | Rare categories unstable                                | Minority categories may carry higher fraud rate                                  |
| **Temporal variable unused**         | fecha                      | Loss of behavioral signal                               | Fraud often time-dependent                                                       |
| **Behavior–amount interaction**      | e, monto                   | Interaction effect ignored                              | Abnormal amount-to-activity ratios may indicate fraud                            |
| **Outliers in behavioral variables** | e, f                       | Risk of removing fraud signal if trimmed                | Fraud often occurs in extreme tails                                              |
| **No duplicates found**              | Dataset                    | None                                                    | Dataset integrity confirmed                                                      |



# List of Original Columns to Keep or Drop



| Variable   | Type     | EDA Findings                          | Action | Justification                                   |
| ---------- | -------- | ------------------------------------- | ------ | ----------------------------------------------- |
| **a**      | Cat      | 85.7% concentrated in one category   | Keep   | Minority categories may carry higher fraud rate |
| **b**      | Numeric  | 8.7% missing                          | Keep   | Missingness may be predictive                   |
| **c**      | Numeric  | 8.7% missing                          | Keep   | Same reasoning as `b`                           |
| **d**      | Numeric  | Correlated with `m`                   | Keep   | Potential predictive signal                     |
| **e**      | Numeric  | 43.4% zeros, extreme skew             | Keep   | Strong behavioral signal candidate              |
| **f**      | Numeric  | 16.9% zeros, highly skewed            | Keep   | Tail behavior likely informative                |
| **g**      | Cat      | Country-like variable                 | Keep   | Geographic fraud patterns common                |
| **h**      | Numeric  | 8.6% zeros                            | Keep   | Zero may indicate inactivity                    |
| **j**      | Cat      | Low cardinality (5.5% distinct values) | Keep   | Few categories may carry fraud signal           |
| **k**      | Numeric  | 100% unique                           | Drop   | Identifier; leakage risk                        |
| **l**      | Numeric  | Correlated with `f`                   | Keep   | Potential predictive contribution               |
| **m**      | Numeric  | 10.6% zeros, correlated with `d`      | Keep   | Behavioral indicator                            |
| **n**      | Cat      | Binary-looking                        | Keep   | Model-ready                                     |
| **o**      | Numeric  | 72.6% missing                         | Keep   | Missingness likely predictive                   |
| **p**      | Cat      | Binary categorical                    | Keep   | Risk segmentation feature                       |
| **fecha**  | DateTime | Timestamp                             | Keep   | Temporal fraud patterns                         |
| **monto**  | Numeric  | Heavy-tailed distribution             | Keep   | Core economic driver                            |
| **score**  | Numeric  | Pre-existing score                    | Keep   | Likely strong predictor (check leakage)         |
| **fraude** | Binary   | 5% positive                           | Keep   | Target                                          |




# Next Steps

Based on the main highlights and the data behaviors detected during the initial exploration, the strategy for Phase 2 has been updated to focus on high-impact feature preparation:

### Phase 2: Feature Engineering & Target Analysis 

* **Technical Literature Review:** Research and select specific preprocessing techniques (e.g., handling extreme skewness, high cardinality, and missingness) identified during the Phase 1 exploration.
* **Feature Prioritization:** Compile a list of potential new variables and rank them by their expected predictive power and implementation complexity.
* **Feature Engineering:** Derive new features based on the initial hypotheses (such as temporal extractions from `fecha` or behavioral ratios) to enhance the model's predictive performance.
* **Target Characterization:** Analyze the distribution of the `fraude` variable and its specific relationship with other variables.
* **Variable Relevance:** Use statistical tests (e.g., Chi-Squared for categorical data, ANOVA for numerical data) to quantify the relationship between features and the target.
* **Hypothesis Formulation:** Create a list of "Risk Indicators" (e.g., "Transactions above amount $X$ are $Y\%$ more likely to be fraud").
* **Feature Selection:** If necessary, other feature selection could be done after this previous steps. 

---

The other Phases continue planned as previous. 

### Phase 3: Model Selection & Profit Optimization
* **Algorithm Candidates:** Train high-performance classifiers (e.g., Random Forest, XGBoost) and establish a baseline using Logistic Regression.
* **Cost-Sensitive Learning:** Incorporate the 4:1 loss ratio ($100\%$ loss vs. $25\%$ margin) directly into the model training weights or cost function.

### Phase 4: Evaluation & Business Calibration
* **Threshold Optimization:** Instead of using the default 0.5 threshold, identify the specific probability cutoff (theoretically **0.20**) that maximizes the Profit Function.
* **Model Calibration:** Ensure that the predicted probabilities accurately reflect the real-world likelihood of fraud so that the thresholding logic remains robust.

**Iterative Refinement:** At the conclusion of each phase, the strategy will be updated based on data findings. The final output will be a production-ready ML model optimized for **Maximum Net Profit**.